# Task 3: Model Validation, Overfitting Control & Hyperparameter Tuning

This notebook demonstrates model validation, overfitting detection, cross-validation, hyperparameter tuning, and model selection on the California Housing regression dataset.

## Objective

Build a regression workflow that compares baseline and tuned models using train/test evaluation, K-Fold cross-validation, RMSE, and R². The main tuning example uses `DecisionTreeRegressor` because tree depth and split settings clearly show overfitting control.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, validation_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42


## 1. Load and Inspect the Data

In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
display(df.head())
print('Shape:', df.shape)
print('Missing values:', int(df.isna().sum().sum()))
display(df.describe().T)


In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(df.corr(numeric_only=True), cmap='RdBu_r', center=0, annot=True, fmt='.2f')
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()


## 2. Train/Test Split

The holdout test set is kept aside until model comparison. This prevents us from tuning decisions directly on the final evaluation data.

In [ ]:
X = df.drop(columns='MedHouseVal')
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print('Train size:', X_train.shape)
print('Test size:', X_test.shape)


## 3. Helper Function for Evaluation

In [ ]:
def evaluate_model(name, model, X_train, y_train, X_test, y_test):
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)
    return {
        'Model': name,
        'Train RMSE': mean_squared_error(y_train, train_pred, squared=False),
        'Test RMSE': mean_squared_error(y_test, test_pred, squared=False),
        'Train R2': r2_score(y_train, train_pred),
        'Test R2': r2_score(y_test, test_pred),
        'Test MAE': mean_absolute_error(y_test, test_pred),
    }


## 4. Baseline Models

A Linear Regression baseline is stable but may underfit non-linear patterns. An unconstrained Decision Tree often overfits by memorizing the training set.

In [ ]:
linear_model = Pipeline(steps=[
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])
linear_model.fit(X_train, y_train)

overfit_tree = DecisionTreeRegressor(random_state=RANDOM_STATE)
overfit_tree.fit(X_train, y_train)

baseline_results = pd.DataFrame([
    evaluate_model('Linear Regression', linear_model, X_train, y_train, X_test, y_test),
    evaluate_model('Unconstrained Decision Tree', overfit_tree, X_train, y_train, X_test, y_test),
])
display(baseline_results)


### Overfitting Check

A model is likely overfitting when training performance is much better than test performance. For an unconstrained tree, train RMSE is often close to zero while test RMSE remains much larger.

In [ ]:
baseline_results.set_index('Model')[['Train RMSE', 'Test RMSE']].plot(kind='bar', figsize=(9, 5))
plt.title('Train vs Test RMSE Reveals Overfitting')
plt.ylabel('RMSE')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()


## 5. Cross-Validation

Cross-validation gives a more reliable estimate than a single train/test split because the model is evaluated across multiple folds.

In [ ]:
cv_models = {
    'Linear Regression': linear_model,
    'Decision Tree (default)': DecisionTreeRegressor(random_state=RANDOM_STATE),
    'Decision Tree (depth=6)': DecisionTreeRegressor(max_depth=6, random_state=RANDOM_STATE),
}

cv_summary = []
for name, model in cv_models.items():
    neg_rmse_scores = cross_val_score(
        model, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error'
    )
    r2_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    cv_summary.append({
        'Model': name,
        'CV RMSE Mean': -neg_rmse_scores.mean(),
        'CV RMSE Std': neg_rmse_scores.std(),
        'CV R2 Mean': r2_scores.mean(),
        'CV R2 Std': r2_scores.std(),
    })

cv_summary = pd.DataFrame(cv_summary).sort_values('CV RMSE Mean')
display(cv_summary)


## 6. Validation Curve for Tree Depth

The validation curve shows the bias-variance tradeoff. Shallow trees can underfit, while very deep trees can overfit.

In [ ]:
depth_range = np.arange(1, 21)
train_scores, valid_scores = validation_curve(
    DecisionTreeRegressor(random_state=RANDOM_STATE),
    X_train, y_train,
    param_name='max_depth',
    param_range=depth_range,
    cv=5,
    scoring='neg_root_mean_squared_error'
)

train_rmse = -train_scores.mean(axis=1)
valid_rmse = -valid_scores.mean(axis=1)

plt.figure(figsize=(9, 5))
plt.plot(depth_range, train_rmse, marker='o', label='Training RMSE')
plt.plot(depth_range, valid_rmse, marker='o', label='Validation RMSE')
plt.xlabel('max_depth')
plt.ylabel('RMSE')
plt.title('Validation Curve: Decision Tree Depth')
plt.legend()
plt.tight_layout()
plt.show()


## 7. Hyperparameter Tuning with GridSearchCV

`GridSearchCV` tests combinations of hyperparameters using cross-validation and selects the settings with the best validation score.

In [ ]:
param_grid = {
    'max_depth': [3, 5, 7, 9, 12, None],
    'min_samples_split': [2, 10, 25, 50],
    'min_samples_leaf': [1, 5, 10, 20],
}

grid_search = GridSearchCV(
    estimator=DecisionTreeRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    scoring='neg_root_mean_squared_error',
    cv=5,
    n_jobs=-1,
    return_train_score=True
)

grid_search.fit(X_train, y_train)

print('Best parameters:', grid_search.best_params_)
print('Best CV RMSE:', -grid_search.best_score_)


In [ ]:
tuning_results = pd.DataFrame(grid_search.cv_results_)
cols = [
    'param_max_depth', 'param_min_samples_split', 'param_min_samples_leaf',
    'mean_train_score', 'mean_test_score', 'std_test_score', 'rank_test_score'
]
summary = tuning_results[cols].copy()
summary['mean_train_RMSE'] = -summary['mean_train_score']
summary['mean_cv_RMSE'] = -summary['mean_test_score']
summary = summary.drop(columns=['mean_train_score', 'mean_test_score']).sort_values('rank_test_score')
display(summary.head(10))


## 8. Final Model Comparison on the Test Set

After tuning is complete, evaluate the selected model once on the holdout test set.

In [ ]:
best_tree = grid_search.best_estimator_

comparison = pd.DataFrame([
    evaluate_model('Linear Regression', linear_model, X_train, y_train, X_test, y_test),
    evaluate_model('Unconstrained Decision Tree', overfit_tree, X_train, y_train, X_test, y_test),
    evaluate_model('Tuned Decision Tree', best_tree, X_train, y_train, X_test, y_test),
])

display(comparison.sort_values('Test RMSE'))


In [ ]:
final_pred = best_tree.predict(X_test)
residuals = y_test - final_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(x=y_test, y=final_pred, alpha=0.35, ax=axes[0], color='#2f7fd0')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')
axes[0].set_title('Tuned Tree: Actual vs Predicted')
axes[0].set_xlabel('Actual MedHouseVal')
axes[0].set_ylabel('Predicted MedHouseVal')

sns.scatterplot(x=final_pred, y=residuals, alpha=0.35, ax=axes[1], color='#555555')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Tuned Tree Residuals')
axes[1].set_xlabel('Predicted MedHouseVal')
axes[1].set_ylabel('Residual')
plt.tight_layout()
plt.show()


## 9. Optional: Random Forest Comparison

A Random Forest can reduce variance by averaging many trees. It is included as a stronger comparison model, but the main Task 3 tuning workflow above uses Decision Tree hyperparameters for clarity.

In [ ]:
forest = RandomForestRegressor(
    n_estimators=150,
    max_depth=18,
    min_samples_leaf=2,
    random_state=RANDOM_STATE,
    n_jobs=-1
)
forest.fit(X_train, y_train)
forest_result = evaluate_model('Random Forest comparison', forest, X_train, y_train, X_test, y_test)
display(pd.DataFrame([forest_result]))


## 10. Final Justification

- The unconstrained Decision Tree shows overfitting when train error is much lower than test error.
- Cross-validation gives a more dependable estimate than a single split.
- `GridSearchCV` controls overfitting by selecting settings such as `max_depth`, `min_samples_split`, and `min_samples_leaf`.
- The final model should be selected based on cross-validation performance and confirmed once on the holdout test set using RMSE and R².
- If deployment performance matters, compare the tuned tree with ensemble methods such as Random Forest or Gradient Boosting.